In [1]:
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import rasterio
import xarray as xr
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import boto3

import os
os.environ['AWS_PROFILE'] = 'ctrees'
os.environ['GDAL_DISABLE_READDIR_ON_OPEN'] = 'YES'
os.environ['VSI_CURL_ALLOW_CREDENTIALS'] = 'YES'


In [2]:
feature_names = [f'emb_{i:03d}' for i in range(128)]

In [3]:
# get all s3 files in training_parquet_prefix
s3 = boto3.client('s3')
bucket_name = 'ctrees-development'
prefix = 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/'
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.parquet')]
print(files)

# load all parquet files into a single dataframe
dfs = []
for file in files:
    path = f's3://{bucket_name}/{file}'
    df = pd.read_parquet(path, storage_options={'profile': 'ctrees'})
    dfs.append(df)
df = pd.concat(dfs, ignore_index=True)

['activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-54.35_-11.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-54.35_-11.75_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-54.35_-11.85_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-54.45_-11.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-54.45_-11.75_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-54.45_-11.85_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-55.85_-3.55_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-55.85_-3.65_samples.parquet', 'activity/tessera/logging_pantropical_v1/extracted_embeddings_reinforced_samples/-55.95_-3.55_sam

In [4]:
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Split with stratification
X_train, X_test = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df['class']
)

# Balance training set: undersample class 0 to match class 1
n_class1_train = (X_train['class'] == 1).sum()
class0_train_idx = X_train[X_train['class'] == 0].sample(n=n_class1_train, random_state=42).index
class1_train_idx = X_train[X_train['class'] == 1].index
X_train = X_train.loc[class0_train_idx.union(class1_train_idx)].sample(frac=1, random_state=42)

print(f"Training set after balancing: class 0 = {(X_train['class']==0).sum()}, class 1 = {(X_train['class']==1).sum()}")
print(f"Test set (original distribution): class 0 = {(X_test['class']==0).sum()}, class 1 = {(X_test['class']==1).sum()}")

train_data = lgb.Dataset(X_train[feature_names], label=X_train['class'])
test_data = lgb.Dataset(X_test[feature_names], label=X_test['class'], reference=train_data)

param_grid = {
    'num_leaves': [31, 63, 127],
    'min_child_samples': [10, 20, 50],
}

best_score = float('inf')
best_params = None
best_num_boost_round = None

for grid_params in ParameterGrid(param_grid):
    params = {
        'objective': 'binary',
        'metric': ['binary_logloss', 'auc'],
        'verbosity': -1,
        'bagging_freq': 1,
        'force_col_wise': True,
        'learning_rate': 0.1,
        **grid_params
    }

    cv_results = lgb.cv(
        params,
        train_data,
        nfold=3,
        num_boost_round=500,
        stratified=True,
        seed=42,
        callbacks=[lgb.early_stopping(30)]
    )

    mean_score = min(cv_results['valid binary_logloss-mean'])
    best_iter = np.argmin(cv_results['valid binary_logloss-mean']) + 1

    if mean_score < best_score:
        best_score = mean_score
        best_params = params
        best_num_boost_round = best_iter

print("Best params:", best_params)
print("Best CV logloss:", best_score)
print("Best num_boost_round:", best_num_boost_round)

bst = lgb.train(
    best_params,
    train_data,
    num_boost_round=best_num_boost_round,
    valid_sets=[test_data],
    callbacks=[lgb.early_stopping(50)]
)

y_pred_prob = bst.predict(X_test[feature_names], num_iteration=bst.best_iteration)
y_pred = (y_pred_prob >= 0.5).astype(int)

accuracy = accuracy_score(X_test['class'], y_pred)
auc = roc_auc_score(X_test['class'], y_pred_prob)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'AUC: {auc:.4f}')
print(classification_report(X_test['class'], y_pred))

Training set after balancing: class 0 = 95656, class 1 = 95656
Test set (original distribution): class 0 = 50250, class 1 = 16881
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.0579493 + 0.00135815	valid's auc: 0.997957 + 0.000126015
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.0523532 + 0.0017385	valid's auc: 0.998248 + 0.000124966
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[371]	valid's binary_logloss: 0.0555834 + 0.00222478	valid's auc: 0.998123 + 0.000121555
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's binary_logloss: 0.0579378 + 0.00109709	valid's auc: 0.997961 + 0.000112611
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Be

In [5]:
model_name = 'pantropical_v1_reinforced.pkl'
joblib.dump(bst, model_name)

# save in s3
s3 = boto3.client('s3')
bucket_name = 'ctrees-development'
object_name = f'activity/tessera/logging_pantropical_v1/model/{model_name}'
s3.upload_file(model_name, bucket_name, object_name)